In [1]:
import gc
import time
import joblib
import warnings
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings('ignore')

# --- ПУТИ ---
BASE_DIR = Path(r"C:\Users\egor0\ProjectsVSCode\ML_Practice")
DATA_DIR = BASE_DIR / "data"
CLEAN_DIR = DATA_DIR / "clean"
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)
ENGINE = 'fastparquet'
RANDOM_STATE = 42

cat_cols_present = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo', 
    'id_12', 'id_15', 'id_16', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 
    'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 
    'id_36', 'id_37', 'id_38'
]

print("Загрузка данных...")
df_train = pd.read_parquet(CLEAN_DIR / 'train_filtered.parquet', engine=ENGINE)
df_val = pd.read_parquet(CLEAN_DIR / 'val_filtered.parquet', engine=ENGINE)

X_train_full = df_train.drop(columns=['isFraud'])
y_train_full = df_train['isFraud']
X_val = df_val.drop(columns=['isFraud'])
y_val = df_val['isFraud']
del df_train, df_val
gc.collect()

print(f"Исходный train: {len(X_train_full):,}, мошенничеств: {y_train_full.mean()*100:.2f}%")
X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full, 
    train_size=180000, 
    stratify=y_train_full, 
    random_state=RANDOM_STATE
)
print(f"Сэмплированный train: {len(X_train):,}, мошенничеств: {y_train.mean()*100:.2f}%")

del X_train_full, y_train_full
gc.collect()

# Общие результаты
results = {}
history = {}

# Категориальные колонки
cat_cols_actual = [c for c in cat_cols_present if c in X_train.columns]

Загрузка данных...
Исходный train: 419,221, мошенничеств: 3.30%
Сэмплированный train: 180,000, мошенничеств: 3.30%


In [2]:
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

# Приводим категории к типу category
for col in cat_cols_actual:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_val_lgb[col] = X_val_lgb[col].astype('category')

train_data = lgb.Dataset(X_train_lgb, label=y_train, categorical_feature=cat_cols_actual, free_raw_data=True)
val_data = lgb.Dataset(X_val_lgb, label=y_val, reference=train_data, categorical_feature=cat_cols_actual, free_raw_data=True)

params = {
    'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05,
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
    'verbosity': -1, 'seed': RANDOM_STATE, 'num_threads': 4
}

print("Обучение LightGBM")
start = time.time()
evals_result = {}
lgbm_model = lgb.train(
    params, train_data, num_boost_round=1000, valid_sets=[val_data], valid_names=['val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
        lgb.record_evaluation(evals_result)
    ]
)
print(f"Обучение завершено за {time.time() - start:.1f}с")

lgbm_preds = lgbm_model.predict(X_val_lgb)
results['LightGBM'] = {
    'ROC-AUC': roc_auc_score(y_val, lgbm_preds),
    'PR-AUC': average_precision_score(y_val, lgbm_preds),
    'Time': time.time() - start
}
history['LightGBM'] = evals_result

# Сохранение
lgbm_model.save_model(str(MODELS_DIR / 'lightgbm_model.txt'))
print(f"LightGBM сохранен. ROC-AUC: {results['LightGBM']['ROC-AUC']:.4f}\n")

del lgbm_model, train_data, val_data, X_train_lgb, X_val_lgb
gc.collect()

Обучение LightGBM
[100]	val's auc: 0.910517
[200]	val's auc: 0.919486
[300]	val's auc: 0.923679
[400]	val's auc: 0.926067
[500]	val's auc: 0.927207
[600]	val's auc: 0.927346
[700]	val's auc: 0.928697
[800]	val's auc: 0.929673
[900]	val's auc: 0.929787
Обучение завершено за 24.3с
LightGBM сохранен. ROC-AUC: 0.9300



473

In [2]:
# Копируем данные
X_train_lr = X_train.copy()
X_val_lr = X_val.copy()

# One-Hot
X_train_lr = pd.get_dummies(X_train_lr, columns=cat_cols_actual, drop_first=True, dummy_na=True)
X_val_lr = pd.get_dummies(X_val_lr, columns=cat_cols_actual, drop_first=True, dummy_na=True)
X_train_lr, X_val_lr = X_train_lr.align(X_val_lr, join='left', axis=1, fill_value=0)

# Масштабирование
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train_lr)
X_val_lr = scaler.transform(X_val_lr)

# Обучение
print("Обучение LogReg")
start = time.time()
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)
lr_model.fit(X_train_lr, y_train)


lr_preds = lr_model.predict_proba(X_val_lr)[:, 1]
results['LogReg'] = {
    'ROC-AUC': roc_auc_score(y_val, lr_preds),
    'PR-AUC': average_precision_score(y_val, lr_preds),
    'Time': time.time() - start
}

# Сохранение модели и scaler'а
joblib.dump(lr_model, MODELS_DIR / 'logreg_model.joblib')
joblib.dump(scaler, MODELS_DIR / 'logreg_scaler.joblib')
print(f"LogReg сохранен. ROC-AUC: {results['LogReg']['ROC-AUC']:.4f}\n")


Обучение LogReg
LogReg сохранен. ROC-AUC: 0.8527



In [3]:
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=15, class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)

rf_model.fit(X_train_lr, y_train)

rf_preds = rf_model.predict_proba(X_val_lr)[:, 1]
results['RandomForest'] = {
    'ROC-AUC': roc_auc_score(y_val, rf_preds),
    'PR-AUC': average_precision_score(y_val, rf_preds),
    'Time': time.time() - start
}

# Сохранение
joblib.dump(rf_model, MODELS_DIR / 'rf_model.joblib')
print(f"RandomForest сохранен. ROC-AUC: {results['RandomForest']['ROC-AUC']:.4f}\n")

del rf_model, X_train_lr, X_val_lr, scaler
gc.collect()

RandomForest сохранен. ROC-AUC: 0.9047



24